In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

In [3]:
# 1. Load the dataset
print("Loading data...")
df = pd.read_csv('delivery_data.csv')

Loading data...


In [4]:
# Separate Features (X) from the Target Label (y)
X = df.drop('delivery_time_min', axis=1)
y = df['delivery_time_min']

In [5]:
# 2. Split the data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [6]:
# 3. Scale the features (CRITICAL so distance and traffic scale equally)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Build the 3-Layer Neural Network for Regression (Prediction)
model = Sequential([
    Input(shape=(4,)),                      # Layer 1: INPUT LAYER (4 features)
    Dense(16, activation='relu'),           # Layer 2: HIDDEN LAYER (Finds complex patterns)
    Dense(1, activation='linear')           # Layer 3: OUTPUT LAYER (Linear activation for continuous numbers)
])

In [7]:
# 5. Compile the model
# loss='mse' (Mean Squared Error) penalizes big mistakes during training
# metrics=['mae'] (Mean Absolute Error) shows us how many minutes off we are on average
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [8]:
# 6. Train the model
print("\nStarting Training...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    verbose=1
)


Starting Training...
Epoch 1/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 2707.8552 - mae: 49.5926 - val_loss: 2475.7808 - val_mae: 47.3128
Epoch 2/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2682.2446 - mae: 49.3365 - val_loss: 2451.0483 - val_mae: 47.0529
Epoch 3/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 2655.9265 - mae: 49.0717 - val_loss: 2425.3037 - val_mae: 46.7816
Epoch 4/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 2628.3184 - mae: 48.7937 - val_loss: 2398.3921 - val_mae: 46.4965
Epoch 5/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2599.2473 - mae: 48.4982 - val_loss: 2369.9705 - val_mae: 46.1953
Epoch 6/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2568.4270 - mae: 48.1867 - val_loss: 2339.7893 - val_mae: 45.8761
Epoch 7/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2535.1907 - mae: 47.8512 - val_loss: 2307.8657 - val_mae: 45.5352
Epoch 8/25
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 2499.8323 - mae: 47.4930 - val_loss: 2273.7896 - va

In [9]:
# 7. Test it with a "Live" Order
print("\n--- ESTIMATING LIVE ORDER DELIVERY TIME ---")
# Simulating a new order: 8.5km away, Traffic level 7, 15 min prep time, driver has 2 years experience
live_order = pd.DataFrame({
    'distance_km': [8.5],
    'traffic_index': [7],
    'prep_time_min': [15.0],
    'courier_exp_yrs': [2.0]
})

# Scale the live order using the same scaler
live_order_scaled = scaler.transform(live_order)

# Make the prediction
predicted_time = model.predict(live_order_scaled)

print(f"Estimated Delivery Time: {predicted_time[0][0]:.0f} minutes")


--- ESTIMATING LIVE ORDER DELIVERY TIME ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
Estimated Delivery Time: 12 minutes
